## 1

In [3]:
# ============================================================
# HVLT FOR HINDI HANDWRITTEN WORD RECOGNITION
# FULL SINGLE-CELL TRAINING NOTEBOOK (FIXED VERSION)
# ============================================================

# ============================================================
# INSTALLS (RUN ONCE IF NEEDED)
# ============================================================

# !pip install timm transformers jiwer opencv-python pillow tqdm scikit-learn

# ============================================================
# IMPORTS
# ============================================================

import os
import cv2
import time
import random
import unicodedata
import numpy as np

from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from torch.amp import autocast, GradScaler

import timm

from transformers import XLMRobertaModel

from jiwer import wer

# ============================================================
# CONFIG
# ============================================================

ROOT_DIR = "/home/mca/Shweta/handwritten text detection/dataset/Word_Level_Bengali_Training_Set/Word_Level_Training_Set"

TRAIN_TXT = os.path.join(ROOT_DIR, "train.txt")

IMG_HEIGHT = 32
IMG_WIDTH  = 256

BATCH_SIZE = 32

LR = 5e-5

MAX_EPOCHS = 20

MAX_SEQ_LEN = 60

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NUM_WORKERS = 4

SEED = 42

USE_AMP = True

# ============================================================
# SEED
# ============================================================

def set_seed(seed=42):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

# ============================================================
# BENGALI VOCABULARY
# ============================================================

BENGALI_CHARS = (
    "অআইঈউঊঋএঐওঔ"
    "কখগঘঙচছজঝঞ"
    "টঠডঢণতথদধন"
    "পফবভমযরলশষসহড়ঢ়য়"
    "ািীুূৃেৈোৌংঃঁ্"
    "০১২৩৪৫৬৭৮৯"
    "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    ".,!?()[]{}:;-_'/\\&%$#@+*= "
)

SPECIAL_TOKENS = [
    "<PAD>",
    "<SOS>",
    "<EOS>",
    "<UNK>",
]

ALL_TOKENS = SPECIAL_TOKENS + list(BENGALI_CHARS)

char2idx = {
    c: i for i, c in enumerate(ALL_TOKENS)
}

idx2char = {
    i: c for c, i in char2idx.items()
}

VOCAB_SIZE = len(ALL_TOKENS)

PAD_IDX = char2idx["<PAD>"]
SOS_IDX = char2idx["<SOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

print("VOCAB SIZE:", VOCAB_SIZE)

# ============================================================
# LOAD DATA
# ============================================================

samples = []

with open(TRAIN_TXT, "r", encoding="utf-8") as f:

    for line in f:

        line = line.strip()

        if len(line) == 0:
            continue

        parts = line.split("\t")

        if len(parts) != 2:
            continue

        rel_path, text = parts

        text = unicodedata.normalize(
            "NFC",
            text.strip()
        )

        img_path = os.path.join(
            ROOT_DIR,
            rel_path
        )

        if os.path.exists(img_path):

            samples.append(
                (img_path, text)
            )

print("TOTAL SAMPLES:", len(samples))

# ============================================================
# SPLIT DATASET
# ============================================================

train_samples, temp_samples = train_test_split(
    samples,
    test_size=0.15,
    random_state=SEED,
)

val_samples, test_samples = train_test_split(
    temp_samples,
    test_size=0.5,
    random_state=SEED,
)

print("TRAIN:", len(train_samples))
print("VAL:", len(val_samples))
print("TEST:", len(test_samples))

# ============================================================
# IMAGE PREPROCESS
# ============================================================

def preprocess_image(img_path):

    img = cv2.imread(img_path)

    # --------------------------------------------------------
    # CORRUPTED IMAGE CHECK
    # --------------------------------------------------------

    if img is None:

        print(f"[WARNING] Corrupted image: {img_path}")

        img = np.ones(
            (IMG_HEIGHT, IMG_WIDTH, 3),
            dtype=np.uint8
        ) * 255

    else:

        h, w = img.shape[:2]

        # invalid image dimensions
        if h <= 0 or w <= 0:

            print(f"[WARNING] Invalid image size: {img_path}")

            img = np.ones(
                (IMG_HEIGHT, IMG_WIDTH, 3),
                dtype=np.uint8
            ) * 255

        else:

            img = cv2.cvtColor(
                img,
                cv2.COLOR_BGR2RGB
            )

            scale = IMG_HEIGHT / h

            new_w = max(
                1,
                int(w * scale)
            )

            img = cv2.resize(
                img,
                (new_w, IMG_HEIGHT)
            )

            # ------------------------------------------------
            # PAD OR RESIZE
            # ------------------------------------------------

            if new_w < IMG_WIDTH:

                pad = np.ones(
                    (
                        IMG_HEIGHT,
                        IMG_WIDTH - new_w,
                        3
                    ),
                    dtype=np.uint8
                ) * 255

                img = np.concatenate(
                    [img, pad],
                    axis=1
                )

            else:

                img = cv2.resize(
                    img,
                    (IMG_WIDTH, IMG_HEIGHT)
                )

    # --------------------------------------------------------
    # NORMALIZATION
    # --------------------------------------------------------

    img = img.astype(np.float32) / 255.0

    img = (img - 0.5) / 0.5

    img = np.transpose(
        img,
        (2, 0, 1)
    )

    return torch.tensor(
        img,
        dtype=torch.float32
    )
    
    
# ============================================================
# TEXT ENCODING
# ============================================================

def encode_text(text):

    tokens = [SOS_IDX]

    for ch in text:

        tokens.append(
            char2idx.get(ch, UNK_IDX)
        )

    tokens.append(EOS_IDX)

    if len(tokens) < MAX_SEQ_LEN:

        tokens += [PAD_IDX] * (
            MAX_SEQ_LEN - len(tokens)
        )

    else:

        tokens = tokens[:MAX_SEQ_LEN]

        tokens[-1] = EOS_IDX

    return torch.tensor(
        tokens,
        dtype=torch.long
    )

# ============================================================
# TEXT DECODING
# ============================================================

def decode_tokens(tokens):

    chars = []

    for t in tokens:

        t = int(t)

        if t == EOS_IDX:
            break

        if t in [PAD_IDX, SOS_IDX]:
            continue

        chars.append(
            idx2char.get(t, "")
        )

    return "".join(chars)

# ============================================================
# DATASET
# ============================================================

class HindiWordDataset(Dataset):

    def __init__(self, samples):

        self.samples = samples

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        img_path, text = self.samples[idx]

        image = preprocess_image(img_path)

        tokens = encode_text(text)

        return image, tokens, text

# ============================================================
# DATALOADERS
# ============================================================

train_loader = DataLoader(
    HindiWordDataset(train_samples),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

val_loader = DataLoader(
    HindiWordDataset(val_samples),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

test_loader = DataLoader(
    HindiWordDataset(test_samples),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

# ============================================================
# POSITIONAL BRIDGE
# ============================================================

class PositionalBridge(nn.Module):

    def __init__(
        self,
        in_channels=1024,
        d_model=768,
        vis_seq_len=256,
    ):
        super().__init__()

        self.pool = nn.AdaptiveAvgPool2d(
            (1, vis_seq_len)
        )

        self.proj = nn.Linear(
            in_channels,
            d_model,
        )

        self.pos_embed = nn.Parameter(
            torch.randn(
                1,
                vis_seq_len,
                d_model,
            ) * 0.02
        )

    def forward(self, x):

        # x = (B,H,W,C)

        B, H, W, C = x.shape

        # -> (B,C,H,W)
        x = x.permute(0, 3, 1, 2)

        # -> (B,C,1,T)
        x = self.pool(x)

        # -> (B,C,T)
        x = x.squeeze(2)

        # -> (B,T,C)
        x = x.permute(0, 2, 1)

        # 1024 -> 768
        x = self.proj(x)

        x = x + self.pos_embed

        return x

# ============================================================
# DECODER
# ============================================================

class HindiDecoder(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model=768,
        n_heads=12,
        n_layers=6,
    ):
        super().__init__()

        self.token_embed = nn.Embedding(
            vocab_size,
            d_model,
            padding_idx=PAD_IDX,
        )

        self.pos_embed = nn.Embedding(
            MAX_SEQ_LEN,
            d_model,
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=3072,
            dropout=0.1,
            batch_first=True,
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=n_layers,
        )

        self.output_proj = nn.Linear(
            d_model,
            vocab_size,
        )

        print("Loading XLM-RoBERTa...")

        try:

            xlm = XLMRobertaModel.from_pretrained(
                "xlm-roberta-base"
            )

            emb = xlm.embeddings.word_embeddings.weight

            n = min(
                emb.shape[0],
                vocab_size,
            )

            self.token_embed.weight.data[:n] = emb[:n]

            del xlm

            print("Loaded multilingual weights.")

        except Exception as e:

            print("Pretrained load failed:", e)

    def forward(
        self,
        memory,
        tgt_tokens,
    ):

        B, T = tgt_tokens.shape

        pos = torch.arange(
            T,
            device=tgt_tokens.device,
        ).unsqueeze(0).expand(B, -1)

        x = (
            self.token_embed(tgt_tokens)
            + self.pos_embed(pos)
        )

        tgt_mask = torch.triu(
            torch.ones(
                T,
                T,
                device=tgt_tokens.device,
            ),
            diagonal=1,
        ).bool()

        tgt_key_padding_mask = (
            tgt_tokens == PAD_IDX
        )

        out = self.decoder(
            tgt=x,
            memory=memory,
            tgt_mask=tgt_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
        )

        logits = self.output_proj(out)

        return logits

    @torch.no_grad()
    def greedy_decode(
        self,
        memory,
        max_len=MAX_SEQ_LEN,
    ):

        B = memory.shape[0]

        generated = torch.full(
            (B, 1),
            SOS_IDX,
            device=memory.device,
            dtype=torch.long,
        )

        finished = torch.zeros(
            B,
            dtype=torch.bool,
            device=memory.device,
        )

        for _ in range(max_len):

            logits = self.forward(
                memory,
                generated,
            )

            next_token = logits[:, -1].argmax(dim=-1)

            next_token = next_token.masked_fill(
                finished,
                PAD_IDX,
            )

            generated = torch.cat(
                [
                    generated,
                    next_token.unsqueeze(1),
                ],
                dim=1,
            )

            finished |= (
                next_token == EOS_IDX
            )

            if finished.all():
                break

        return generated[:, 1:]

# ============================================================
# HVLT MODEL
# ============================================================

class HVLT(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(32, 256),
            strict_img_size=False,
        )

        self.bridge = PositionalBridge(
            in_channels=1024,
            d_model=768,
            vis_seq_len=256,
        )

        self.decoder = HindiDecoder(
            vocab_size=VOCAB_SIZE
        )

    def forward(
        self,
        images,
        tgt_tokens,
    ):

        # timm already returns (B,H,W,C)
        feats = self.encoder(images)[-1]

        memory = self.bridge(feats)

        logits = self.decoder(
            memory,
            tgt_tokens,
        )

        return logits

    @torch.no_grad()
    def predict(self, images):

        feats = self.encoder(images)[-1]

        memory = self.bridge(feats)

        preds = self.decoder.greedy_decode(memory)

        return preds
    
# ============================================================
# LOSS
# ============================================================

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_IDX
)

# ============================================================
# METRICS
# ============================================================

def char_accuracy(preds, labels):

    correct = 0

    total = 0

    for p, l in zip(preds, labels):

        n = min(len(p), len(l))

        for i in range(n):

            if p[i] == l[i]:
                correct += 1

        total += max(len(p), len(l))

    return 100.0 * correct / max(total, 1)

# ============================================================
# MODEL
# ============================================================

model = HVLT().to(DEVICE)

optimizer = Adam(
    model.parameters(),
    lr=LR,
)

scaler = GradScaler(
    "cuda",
    enabled=USE_AMP
)

print(
    "TOTAL PARAMS:",
    sum(
        p.numel()
        for p in model.parameters()
    ) / 1e6,
    "M"
)

# ============================================================
# TRAINING LOOP
# ============================================================

best_wer = 9999

for epoch in range(1, MAX_EPOCHS + 1):

    # ========================================================
    # TRAIN
    # ========================================================

    model.train()

    train_loss = 0

    train_preds = []

    train_labels = []

    pbar = tqdm(train_loader)

    for images, targets, labels in pbar:

        images = images.to(DEVICE)

        targets = targets.to(DEVICE)

        decoder_input = targets[:, :-1]

        decoder_target = targets[:, 1:]

        optimizer.zero_grad()

        with autocast(
            "cuda",
            enabled=USE_AMP
        ):

            logits = model(
                images,
                decoder_input,
            )

            loss = criterion(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                decoder_target.reshape(-1),
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            5.0,
        )

        scaler.step(optimizer)

        scaler.update()

        train_loss += loss.item()

        with torch.no_grad():

            pred_ids = logits.argmax(dim=-1)

            preds = [
                decode_tokens(x)
                for x in pred_ids
            ]

        train_preds.extend(preds)

        train_labels.extend(labels)

        pbar.set_description(
            f"Train Loss: {loss.item():.4f}"
        )

    train_cer = char_accuracy(
        train_preds,
        train_labels,
    )

    train_wer = wer(
        train_labels,
        train_preds,
    ) * 100

    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    val_loss = 0

    val_preds = []

    val_labels = []

    with torch.no_grad():

        pbar = tqdm(val_loader)

        for images, targets, labels in pbar:

            images = images.to(DEVICE)

            targets = targets.to(DEVICE)

            decoder_input = targets[:, :-1]

            decoder_target = targets[:, 1:]

            logits = model(
                images,
                decoder_input,
            )

            loss = criterion(
                logits.reshape(
                    -1,
                    VOCAB_SIZE
                ),
                decoder_target.reshape(-1),
            )

            val_loss += loss.item()

            pred_ids = model.predict(images)

            preds = [
                decode_tokens(x)
                for x in pred_ids
            ]

            val_preds.extend(preds)

            val_labels.extend(labels)

    val_cer = char_accuracy(
        val_preds,
        val_labels,
    )

    val_wer = wer(
        val_labels,
        val_preds,
    ) * 100

    # ========================================================
    # LOGGING
    # ========================================================

    print("\n")
    print("=" * 60)

    print(f"EPOCH {epoch}")

    print(
        f"TRAIN LOSS: {train_loss / len(train_loader):.4f}"
    )

    print(
        f"VAL LOSS: {val_loss / len(val_loader):.4f}"
    )

    print(
        f"TRAIN CAR: {train_cer:.2f}%"
    )

    print(
        f"VAL CAR: {val_cer:.2f}%"
    )

    print(
        f"TRAIN WER: {train_wer:.2f}%"
    )

    print(
        f"VAL WER: {val_wer:.2f}%"
    )

    print("=" * 60)

    # ========================================================
    # SAVE BEST
    # ========================================================

    if val_wer < best_wer:

        best_wer = val_wer

        torch.save(
            {
                "model": model.state_dict(),
                "epoch": epoch,
                "val_wer": val_wer,
            },
            "best_bengali_hvlt.pth",
        )

        print("BEST MODEL SAVED.")

# ============================================================
# TEST EVALUATION
# ============================================================

print("\nFINAL TEST EVALUATION")

model.eval()

test_preds = []

test_labels = []

with torch.no_grad():

    for images, targets, labels in tqdm(test_loader):

        images = images.to(DEVICE)

        pred_ids = model.predict(images)

        preds = [
            decode_tokens(x)
            for x in pred_ids
        ]

        test_preds.extend(preds)

        test_labels.extend(labels)

test_cer = char_accuracy(
    test_preds,
    test_labels,
)

test_wer = wer(
    test_labels,
    test_preds,
) * 100

print("\nTEST RESULTS")

print("TEST CAR:", test_cer)

print("TEST WER:", test_wer)

# ============================================================
# SAMPLE PREDICTIONS
# ============================================================

print("\nSAMPLE PREDICTIONS\n")

for i in range(10):

    print("GT   :", test_labels[i])

    print("PRED :", test_preds[i])

    print("-" * 50)

VOCAB SIZE: 155
TOTAL SAMPLES: 79663
TRAIN: 67713
VAL: 5975
TEST: 5975
Loading XLM-RoBERTa...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded multilingual weights.
TOTAL PARAMS: 144.671283 M


100%|████████████████████████████████████████████████████████████████████| 187/187 [01:10<00:00,  2.65it/s]


...



EPOCH 19
TRAIN LOSS: 0.0350
VAL LOSS: 0.3718
TRAIN CAR: 93.83%
VAL CAR: 76.55%
TRAIN WER: 9.56%
VAL WER: 30.26%


100%|████████████████████████████████████████████████████████████████████| 187/187 [01:04<00:00,  2.91it/s]




EPOCH 20
TRAIN LOSS: 0.0358
VAL LOSS: 0.3563
TRAIN CAR: 93.85%
VAL CAR: 76.94%
TRAIN WER: 9.67%
VAL WER: 29.42%

FINAL TEST EVALUATION


100%|████████████████████████████████████████████████████████████████████| 187/187 [00:52<00:00,  3.54it/s]



TEST RESULTS
TEST CAR: 77.308753803694
TEST WER: 29.255230125523013

SAMPLE PREDICTIONS

GT   : ।
PRED : <UNK>
--------------------------------------------------
GT   : করে
PRED : করো
--------------------------------------------------
GT   : যেতে
PRED : যেতে
--------------------------------------------------
GT   : পৃষ্ঠা
PRED : পৃষ্ঠা
--------------------------------------------------
GT   : নেই
PRED : নেই
--------------------------------------------------
GT   : হওয়ার
PRED : হওয়ার
--------------------------------------------------
GT   : ঠিক
PRED : ঠিক
--------------------------------------------------
GT   : যায়
PRED : যায়
--------------------------------------------------
GT   : আমরা
PRED : আমরা
--------------------------------------------------
GT   : তাকে
PRED : তাকে
--------------------------------------------------


## 2 gpt

In [1]:
# =====================================================================
# BENGALI HANDWRITTEN WORD RECOGNITION – FULL GPT‑2 PIPELINE
# (Swin + TPS + BiLSTM + GPT‑2 Decoder + LLRD + Agentic Verification)
# =====================================================================

import os
import cv2
import random
import unicodedata
import numpy as np
from collections import defaultdict

from tqdm import tqdm
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import autocast, GradScaler

import timm
from transformers import GPT2LMHeadModel, GPT2Config

import Levenshtein
from jiwer import wer

os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ============================================================
# 1. CONFIGURATION
# ============================================================

# --- CHANGE THIS TO YOUR BENGALI DATASET PATH ---
ROOT_DIR = "/home/mca24-26/handwritten text detection/dataset/Word_Level_Bengali_Training_Set/Word_Level_Training_Set"
TRAIN_TXT = os.path.join(ROOT_DIR, "train.txt")
CHECKPOINT_PATH = "./best_bengali_htr_gpt.pth"

IMG_HEIGHT = 32          # Keep same as your original
IMG_WIDTH  = 256
MAX_SEQ_LEN = 60         # Bengali words may be longer, keep 60 as in your code
BEAM_SIZE = 5            # For final testing only
D_MODEL = 768
BATCH_SIZE = 32          # Adjust based on GPU memory
LR = 5e-5
MAX_EPOCHS = 30          # You can increase if needed
EARLY_STOPPING_PATIENCE = 8

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_WORKERS = 4
SEED = 42
USE_AMP = True

# ============================================================
# 2. SEED
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

# ============================================================
# 3. BENGALI VOCABULARY & TOKENIZER (from your original code)
# ============================================================

BENGALI_CHARS = (
    "অআইঈউঊঋএঐওঔ"
    "কখগঘঙচছজঝঞ"
    "টঠডঢণতথদধন"
    "পফবভমযরলশষসহড়ঢ়য়"
    "ািীুূৃেৈোৌংঃঁ্"
    "০১২৩৪৫৬৭৮৯"
    "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ"
    ".,!?()[]{}:;-_'/\\&%$#@+*= "
)

SPECIAL_TOKENS = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"]
ALL_TOKENS = SPECIAL_TOKENS + list(BENGALI_CHARS)
char2idx = {c: i for i, c in enumerate(ALL_TOKENS)}
idx2char = {i: c for c, i in char2idx.items()}

VOCAB_SIZE = len(ALL_TOKENS)
PAD_IDX = char2idx["<PAD>"]
SOS_IDX = char2idx["<SOS>"]
EOS_IDX = char2idx["<EOS>"]
UNK_IDX = char2idx["<UNK>"]

print("VOCAB SIZE:", VOCAB_SIZE)

def encode_text(text):
    tokens = [SOS_IDX]
    for ch in text:
        tokens.append(char2idx.get(ch, UNK_IDX))
    tokens.append(EOS_IDX)
    if len(tokens) < MAX_SEQ_LEN:
        tokens += [PAD_IDX] * (MAX_SEQ_LEN - len(tokens))
    else:
        tokens = tokens[:MAX_SEQ_LEN]
        tokens[-1] = EOS_IDX
    return torch.tensor(tokens, dtype=torch.long)

def decode_tokens(tokens):
    chars = []
    for t in tokens:
        t = int(t)
        if t == EOS_IDX:
            break
        if t in [PAD_IDX, SOS_IDX]:
            continue
        chars.append(idx2char.get(t, ""))
    return "".join(chars)

# ============================================================
# 4. IMAGE PREPROCESSING (robust, same as your original)
# ============================================================

def preprocess_image(img_path):
    img = cv2.imread(img_path)
    if img is None:
        # Fallback to blank image
        img = np.ones((IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.uint8) * 255
    else:
        h, w = img.shape[:2]
        if h <= 0 or w <= 0:
            img = np.ones((IMG_HEIGHT, IMG_WIDTH, 3), dtype=np.uint8) * 255
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            scale = IMG_HEIGHT / h
            new_w = max(1, int(w * scale))
            img = cv2.resize(img, (new_w, IMG_HEIGHT))
            if new_w < IMG_WIDTH:
                pad = np.ones((IMG_HEIGHT, IMG_WIDTH - new_w, 3), dtype=np.uint8) * 255
                img = np.concatenate([img, pad], axis=1)
            else:
                img = cv2.resize(img, (IMG_WIDTH, IMG_HEIGHT))
    # Normalize
    img = img.astype(np.float32) / 255.0
    img = (img - 0.5) / 0.5
    img = np.transpose(img, (2, 0, 1))
    return torch.tensor(img, dtype=torch.float32)

# ============================================================
# 5. DATASET
# ============================================================

class BengaliWordDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, text = self.samples[idx]
        image = preprocess_image(img_path)
        tokens = encode_text(text)
        return image, tokens, text

# ============================================================
# 6. LOAD & SPLIT DATA
# ============================================================

samples = []
with open(TRAIN_TXT, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split("\t")
        if len(parts) != 2:
            continue
        rel_path, text = parts
        text = unicodedata.normalize("NFC", text.strip())
        img_path = os.path.join(ROOT_DIR, rel_path)
        if os.path.exists(img_path):
            samples.append((img_path, text))

print("TOTAL SAMPLES:", len(samples))

train_samples, temp_samples = train_test_split(samples, test_size=0.15, random_state=SEED)
val_samples, test_samples = train_test_split(temp_samples, test_size=0.5, random_state=SEED)
print("TRAIN:", len(train_samples), "VAL:", len(val_samples), "TEST:", len(test_samples))

# ============================================================
# 7. DATALOADERS
# ============================================================

train_loader = DataLoader(BengaliWordDataset(train_samples), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(BengaliWordDataset(val_samples),   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(BengaliWordDataset(test_samples),  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# ============================================================
# 8. ARCHITECTURE – TPS + SWIN + BiLSTM + GPT‑2
# ============================================================

class LocalizationNetwork(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.num_control_points = num_control_points
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(True), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(True), nn.AdaptiveAvgPool2d((4, 4))
        )
        self.fc = nn.Sequential(
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(True),
            nn.Linear(256, num_control_points * 2)
        )
        self.fc[-1].weight.data.zero_()
        self.fc[-1].bias.data.zero_()

    def forward(self, x):
        B = x.size(0)
        pts = self.fc(self.conv(x).view(B, -1))
        return pts.view(B, self.num_control_points, 2)

class TPSSpatialTransformer(nn.Module):
    def __init__(self, num_control_points=16):
        super().__init__()
        self.localization = LocalizationNetwork(num_control_points)

    def forward(self, x):
        B = x.size(0)
        cp = self.localization(x)
        cx = cp[:, :, 0].mean(dim=1)
        cy = cp[:, :, 1].mean(dim=1)
        theta = torch.zeros(B, 2, 3, device=x.device)
        theta[:, 0, 0] = 1.0
        theta[:, 1, 1] = 1.0
        theta[:, 0, 2] = torch.tanh(cx) * 0.05
        theta[:, 1, 2] = torch.tanh(cy) * 0.05
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False, padding_mode='border')

class SwinEncoder(nn.Module):
    def __init__(self, d_model=768):
        super().__init__()
        self.swin = timm.create_model(
            "swin_base_patch4_window7_224",
            pretrained=True,
            features_only=True,
            img_size=(IMG_HEIGHT, IMG_WIDTH),
            strict_img_size=False,
        )
        self.proj = nn.Linear(512, d_model)   # Swin-B stage3 has 512 channels
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        features = self.swin(x)
        feat = features[-2]          # stage 3
        B, H, W, C = feat.shape
        feat = feat.view(B, H*W, C)
        return self.norm(self.proj(feat))

class GPT2Decoder(nn.Module):
    def __init__(self, vocab_size, d_model=768):
        super().__init__()
        print("Initializing GPT‑2 decoder for Bengali...")
        config = GPT2Config.from_pretrained("gpt2")
        config.add_cross_attention = True
        config.vocab_size = vocab_size
        self.decoder = GPT2LMHeadModel(config)

        # Optionally load pretrained weights for matching layers
        try:
            pretrained = GPT2LMHeadModel.from_pretrained("gpt2")
            pretrained_dict = pretrained.state_dict()
            mismatch_keys = {"transformer.wte.weight", "lm_head.weight"}
            filtered_dict = {k: v for k, v in pretrained_dict.items() if k not in mismatch_keys}
            load_result = self.decoder.load_state_dict(filtered_dict, strict=False)
            print(f"Loaded {len(filtered_dict)} pretrained layers. Missing: {len(load_result.missing_keys)}")
            del pretrained
        except Exception as e:
            print("Could not load pretrained GPT‑2, using random init:", e)

        print("GPT‑2 decoder ready.")

    def forward(self, input_ids, encoder_hidden_states=None, labels=None):
        return self.decoder(
            input_ids=input_ids,
            encoder_hidden_states=encoder_hidden_states,
            labels=labels
        )

class CompleteHTRPipeline(nn.Module):
    def __init__(self, vocab_size, d_model=768, num_control_points=16):
        super().__init__()
        self.tps_stn = TPSSpatialTransformer(num_control_points)
        self.swin_encoder = SwinEncoder(d_model=d_model)
        self.bilstm = nn.LSTM(
            input_size=d_model, hidden_size=d_model//2, num_layers=2,
            batch_first=True, bidirectional=True, dropout=0.3
        )
        self.gpt2_decoder = GPT2Decoder(vocab_size=vocab_size, d_model=d_model)

    def _extract_visual_memory(self, images):
        rectified = self.tps_stn(images)
        swin_feat = self.swin_encoder(rectified)
        memory, _ = self.bilstm(swin_feat)
        return memory.contiguous()

    def forward(self, images, target_ids, criterion=None):
        memory = self._extract_visual_memory(images)
        dec_input = target_ids[:, :-1].clone()
        dec_input = torch.where(dec_input == -100, torch.ones_like(dec_input), dec_input)
        labels = target_ids[:, 1:].clone()

        outputs = self.gpt2_decoder(input_ids=dec_input, encoder_hidden_states=memory)
        logits = outputs.logits

        if criterion is not None:
            return criterion(logits.reshape(-1, logits.size(-1)), labels.reshape(-1))
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), labels.reshape(-1), ignore_index=-100)

    @torch.no_grad()
    def generate(self, images, max_length=MAX_SEQ_LEN, bos_token_id=SOS_IDX, eos_token_id=EOS_IDX, beam_size=1):
        device = images.device
        B = images.size(0)
        memory = self._extract_visual_memory(images)

        if beam_size == 1:
            # Greedy batch decoding (fast)
            generated = torch.full((B, 1), bos_token_id, dtype=torch.long, device=device)
            finished = torch.zeros(B, dtype=torch.bool, device=device)
            for _ in range(max_length - 1):
                out = self.gpt2_decoder(input_ids=generated, encoder_hidden_states=memory)
                next_tokens = out.logits[:, -1, :].argmax(dim=-1, keepdim=True)
                next_tokens = next_tokens.masked_fill(finished.unsqueeze(1), eos_token_id)
                generated = torch.cat([generated, next_tokens], dim=1)
                finished |= (next_tokens.squeeze(1) == eos_token_id)
                if finished.all():
                    break
            return generated[:, 1:]   # strip BOS

        # ========== BEAM SEARCH (full implementation) ==========
        # We copy the beam search from the IAM code for completeness.
        all_results = []
        for b in range(B):
            mem = memory[b:b+1]
            beams = [(0.0, [bos_token_id])]
            completed = []

            for _ in range(max_length - 1):
                candidates = []
                for score, tokens in beams:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                        continue
                    inp = torch.tensor([tokens], dtype=torch.long, device=device)
                    out = self.gpt2_decoder(input_ids=inp, encoder_hidden_states=mem)
                    log_prob = F.log_softmax(out.logits[0, -1], dim=-1)
                    top_lp, top_id = log_prob.topk(beam_size)
                    for lp, tid in zip(top_lp.tolist(), top_id.tolist()):
                        candidates.append((score + lp, tokens + [tid]))

                if not candidates:
                    break

                candidates.sort(key=lambda x: x[0], reverse=True)
                beams = []
                for score, tokens in candidates[:beam_size * 2]:
                    if tokens[-1] == eos_token_id:
                        completed.append((score, tokens))
                    else:
                        beams.append((score, tokens))
                    if len(beams) == beam_size:
                        break

                if not beams:
                    break

            all_beams = completed if completed else beams
            _, best_tokens = max(all_beams, key=lambda x: x[0])
            result = torch.tensor(best_tokens[1:], dtype=torch.long, device=device)
            all_results.append(result)

        max_len = max(r.size(0) for r in all_results)
        padded = torch.full((B, max_len), eos_token_id, dtype=torch.long, device=device)
        for i, r in enumerate(all_results):
            padded[i, :r.size(0)] = r
        return padded

# ============================================================
# 9. LLRD OPTIMIZER
# ============================================================

def build_llrd_optimizer(model, base_lr=LR, decay_factor=0.75, weight_decay=0.05):
    assigned = set()
    def collect(named_params, filter_fn):
        params = []
        for name, param in named_params:
            if id(param) not in assigned and filter_fn(name):
                params.append(param)
                assigned.add(id(param))
        return params
    def collect_all(named_params):
        params = []
        for name, param in named_params:
            if id(param) not in assigned:
                params.append(param)
                assigned.add(id(param))
        return params

    # GPT-2 cross-attention (new layers)
    gpt2_crossattn = collect(model.gpt2_decoder.named_parameters(),
                             lambda n: "crossattention" in n or "cross_attn" in n)
    # GPT-2 rest
    gpt2_rest = collect_all(model.gpt2_decoder.named_parameters())
    bilstm_params = collect_all(model.bilstm.named_parameters())
    swin_proj = collect(model.swin_encoder.named_parameters(),
                        lambda n: n.startswith("proj.") or n.startswith("norm."))
    swin_s4 = collect(model.swin_encoder.swin.named_parameters(),
                      lambda n: "layers_3" in n)
    swin_s3 = collect(model.swin_encoder.swin.named_parameters(),
                      lambda n: "layers_2" in n)
    swin_s2 = collect(model.swin_encoder.swin.named_parameters(),
                      lambda n: "layers_1" in n)
    swin_s1 = collect_all(model.swin_encoder.swin.named_parameters())
    tps_params = collect_all(model.tps_stn.named_parameters())

    lr = [base_lr,
          base_lr * decay_factor,
          base_lr * decay_factor**2,
          base_lr * decay_factor**3,
          base_lr * decay_factor**4,
          base_lr * decay_factor**5,
          base_lr * decay_factor**6,
          base_lr * decay_factor**7,
          base_lr * decay_factor**3]

    groups = [
        (gpt2_crossattn, lr[0], "gpt2_crossattn"),
        (gpt2_rest,      lr[1], "gpt2_rest"),
        (bilstm_params,  lr[2], "bilstm"),
        (swin_proj,      lr[3], "swin_proj"),
        (swin_s4,        lr[4], "swin_stage4"),
        (swin_s3,        lr[5], "swin_stage3"),
        (swin_s2,        lr[6], "swin_stage2"),
        (swin_s1,        lr[7], "swin_stage1"),
        (tps_params,     lr[8], "tps_stn"),
    ]

    param_groups = [{"params": p, "lr": l, "name": n} for p, l, n in groups if len(p) > 0]
    print("\nLLRD Groups:")
    print(f"{'Name':<20} {'LR':>10} {'Params':>10}")
    print("-" * 44)
    for g in param_groups:
        n = sum(p.numel() for p in g["params"])
        print(f"{g['name']:<20} {g['lr']:>10.2e} {n/1e6:>9.2f}M")
    return AdamW(param_groups, weight_decay=weight_decay)

# ============================================================
# 10. AGENTIC VERIFICATION MODULE (lexicon from training)
# ============================================================

class AgenticVerificationModule:
    def __init__(self, train_samples):
        self.lexicon = defaultdict(int)
        for _, text in train_samples:
            for word in text.split():
                # Remove punctuation for matching
                clean = word.strip(".,!?()[]{}:;\"'").lower()
                if clean:
                    self.lexicon[clean] += 1
        self.freq_max = max(self.lexicon.values()) if self.lexicon else 1
        print(f"Lexicon built: {len(self.lexicon)} unique words, max freq {self.freq_max}")

    def verify_and_correct(self, text_output, confidence=None, confidence_threshold=0.85):
        cleaned = text_output.strip().lower()
        if (not cleaned
                or cleaned in self.lexicon
                or len(cleaned) <= 2
                or any(c.isdigit() for c in cleaned)):
            return text_output

        if confidence is not None and confidence >= confidence_threshold:
            return text_output

        best_candidate = None
        best_score = -float('inf')
        target_len = len(cleaned)

        for word, freq in self.lexicon.items():
            if abs(len(word) - target_len) > 2:
                continue
            dist = Levenshtein.distance(cleaned, word)
            if dist > 2:
                continue
            freq_score = freq / self.freq_max
            score = freq_score - (dist * 1.2)
            if score > best_score:
                best_score = score
                best_candidate = word

        if best_candidate is None:
            return text_output

        # Preserve casing (if original had caps)
        if text_output.isupper():
            return best_candidate.upper()
        elif len(text_output) > 0 and text_output[0].isupper():
            return best_candidate.capitalize()
        return best_candidate

# ============================================================
# 11. METRICS & EARLY STOPPING
# ============================================================

def char_accuracy(preds, labels):
    correct = 0
    total = 0
    for p, l in zip(preds, labels):
        n = min(len(p), len(l))
        for i in range(n):
            if p[i] == l[i]:
                correct += 1
        total += max(len(p), len(l))
    return 100.0 * correct / max(total, 1)

def calculate_metrics(preds, targets):
    cer = char_accuracy(preds, targets)
    wer_val = wer(targets, preds) * 100
    return {"CER": cer, "WER": wer_val}

class EarlyStopping:
    def __init__(self, patience=8, min_delta=0.05):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best = float('inf')
        self.early_stop = False

    def __call__(self, metric):
        if metric < self.best - self.min_delta:
            self.best = metric
            self.counter = 0
        else:
            self.counter += 1
            print(f"EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
        return self.early_stop

# ============================================================
# 12. TRAINING LOOP
# ============================================================

def train():
    model = CompleteHTRPipeline(vocab_size=VOCAB_SIZE).to(DEVICE)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"Total parameters: {total_params/1e6:.2f}M")

    # Freeze Swin for first 3 epochs
    for param in model.swin_encoder.swin.parameters():
        param.requires_grad = False

    optimizer = build_llrd_optimizer(model, base_lr=LR, decay_factor=0.75, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-7)

    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=0.1)
    scaler = GradScaler("cuda", enabled=USE_AMP)

    agent_verifier = AgenticVerificationModule(train_samples)
    early_stopper = EarlyStopping(patience=EARLY_STOPPING_PATIENCE, min_delta=0.1)

    best_val_wer = float('inf')
    start_epoch = 1

    for epoch in range(start_epoch, MAX_EPOCHS + 1):
        # Unfreeze Swin after epoch 3
        if epoch == 4:
            for param in model.swin_encoder.swin.parameters():
                param.requires_grad = True
            optimizer = build_llrd_optimizer(model, base_lr=LR, decay_factor=0.75, weight_decay=0.05)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-7)
            print("Swin encoder unfrozen.")

        # ---- Train ----
        model.train()
        train_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{MAX_EPOCHS} [Train]")
        for images, targets, _ in pbar:
            images = images.to(DEVICE)
            targets = targets.to(DEVICE)
            optimizer.zero_grad()
            with autocast("cuda", enabled=USE_AMP):
                loss = model(images, target_ids=targets, criterion=criterion)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        scheduler.step()
        avg_train_loss = train_loss / len(train_loader)

        # ---- Validation ----
        model.eval()
        all_preds, all_labels = [], []
        first_batch_done = False
        with torch.no_grad():
            for images, _, texts in tqdm(val_loader, desc="Validation"):
                images = images.to(DEVICE)
                tokens = model.generate(images, beam_size=1)  # greedy for speed
                preds = [decode_tokens(x) for x in tokens]
                # Apply agentic verification
                verified_preds = [agent_verifier.verify_and_correct(p) for p in preds]
                all_preds.extend(verified_preds)
                all_labels.extend(texts)

                if not first_batch_done:
                    print("\n--- Sample validation predictions ---")
                    for i in range(min(3, len(preds))):
                        print(f"Target: '{texts[i]}' | Pred: '{preds[i]}' -> Verified: '{verified_preds[i]}'")
                    first_batch_done = True

        metrics = calculate_metrics(all_preds, all_labels)
        val_cer = metrics["CER"]
        val_wer = metrics["WER"]

        print(f"\nEpoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val CER: {val_cer:.2f}% | Val WER: {val_wer:.2f}%")

        if val_wer < best_val_wer:
            best_val_wer = val_wer
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_wer': val_wer,
                'cer': val_cer,
            }, CHECKPOINT_PATH)
            print(f"Best model saved (WER: {val_wer:.2f}%)")

        if early_stopper(val_wer):
            print("Early stopping triggered.")
            break

    print("Training finished.")

# ============================================================
# 13. FINAL TEST EVALUATION (with beam search option)
# ============================================================

def test(use_beam_search=True):
    device = DEVICE
    model = CompleteHTRPipeline(vocab_size=VOCAB_SIZE).to(device)
    if os.path.exists(CHECKPOINT_PATH):
        ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
        model.load_state_dict(ckpt['model_state_dict'])
        print(f"Loaded best model from epoch {ckpt.get('epoch', '?')} (WER: {ckpt.get('best_wer', '?')}%)")
    else:
        print("No checkpoint found, using random weights.")
        return

    model.eval()
    test_preds, test_labels = [], []
    with torch.no_grad():
        for images, _, texts in tqdm(test_loader, desc="Testing"):
            images = images.to(device)
            beam = BEAM_SIZE if use_beam_search else 1
            tokens = model.generate(images, beam_size=beam)
            preds = [decode_tokens(x) for x in tokens]
            test_preds.extend(preds)
            test_labels.extend(texts)

    test_cer = char_accuracy(test_preds, test_labels)
    test_wer = wer(test_labels, test_preds) * 100
    print(f"\nTest CER: {test_cer:.2f}% | Test WER: {test_wer:.2f}%")

    print("\nSample predictions:")
    for i in range(min(10, len(test_preds))):
        print(f"GT : {test_labels[i]}")
        print(f"PR : {test_preds[i]}")
        print("-" * 40)

# ============================================================
# 14. MAIN
# ============================================================

if __name__ == "__main__":
    train()
    # After training, run test with beam search (more accurate) or greedy
    test(use_beam_search=True)   # Set False for greedy

/home/mca24-26/handwritten text detection/sbenv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


VOCAB SIZE: 155
TOTAL SAMPLES: 79663
TRAIN: 67713 VAL: 5975 TEST: 5975


/home/mca24-26/handwritten text detection/sbenv/lib/python3.8/site-packages/timm/layers/interpolate.py:47: UserWarning: torch.searchsorted(): input value tensor is non-contiguous, this will lower the performance due to extra data copy when converting non-contiguous tensor to contiguous, please use contiguous input value tensor if possible. This message will only appear once per program. (Triggered internally at ../aten/src/ATen/native/BucketizationUtils.h:32.)
  idx_right = torch.bucketize(x, p)


Initializing GPT‑2 decoder for Bengali...
Loaded 147 pretrained layers. Missing: 98
GPT‑2 decoder ready.
Total parameters: 209.13M

LLRD Groups:
Name                         LR     Params
--------------------------------------------
gpt2_crossattn         5.00e-05     28.37M
gpt2_rest              3.75e-05     85.96M
bilstm                 2.81e-05      7.09M
swin_proj              2.11e-05      0.40M
swin_stage4            1.58e-05     27.29M
swin_stage3            1.19e-05     57.28M
swin_stage2            8.90e-06      1.71M
swin_stage1            6.67e-06      0.40M
tps_stn                2.11e-05      0.63M
Lexicon built: 12011 unique words, max freq 2586


Validation:   1%|▍                                        | 2/187 [00:00<00:36,  5.03it/s]

...


Epoch 30 | Train Loss: 0.8554 | Val CER: 69.96% | Val WER: 34.36%
Best model saved (WER: 34.36%)
Training finished.
Initializing GPT‑2 decoder for Bengali...
Loaded 147 pretrained layers. Missing: 98
GPT‑2 decoder ready.


/tmp/ipykernel_2497727/1957026306.py:657: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CHECKPOINT_PATH, map_location=device)


Loaded best model from epoch 30 (WER: 34.35983263598326%)


Testing: 100%|████████████████████████████████████████| 187/187 [4:18:28<00:00, 82.93s/it]


Test CER: 73.21% | Test WER: 32.12%

Sample predictions:
GT : ।
PR : <UNK>
----------------------------------------
GT : করে
PR : করো
----------------------------------------
GT : যেতে
PR : যেতে
----------------------------------------
GT : পৃষ্ঠা
PR : পৃষ্ঠা
----------------------------------------
GT : নেই
PR : চাই
----------------------------------------
GT : হওয়ার
PR : হওয়ার
----------------------------------------
GT : ঠিক
PR : কিন
----------------------------------------
GT : যায়
PR : যায়
----------------------------------------
GT : আমরা
PR : আমলা
----------------------------------------
GT : তাকে
PR : তাকে
----------------------------------------
